# Pop-up Events - Preprocessing

Builds `data/popup_events_clean.csv`, the file used by `popup_analysis_basic.py` and the Tableau dashboard.

Raw data (Kaggle, not included in this repository - download and place in `data/raw/`):
- `PopUp_Events_KPI_2025.csv` - 2,133 pop-up events, 15 columns

Steps
1. Standardize column names
2. Convert dates
3. Create `revenue_usd` = units_sold × price_usd
4. Create `country` from `city`
5. Fill missing `end_date` = start_date + lease_length_days
6. Create `category` from `product_name`
7. Save

In [ ]:
import pandas as pd
import numpy as np

RAW_EVENTS = "data/raw/PopUp_Events_KPI_2025.csv"
OUTPUT = "data/popup_events_clean.csv"

In [ ]:
# Load the main dataset
df2 = pd.read_csv(RAW_EVENTS).copy()
df2.head()

## 1. Standardize column names

In [ ]:
def clean_columns(df):
    # Lowercase all column names and strip spaces
    df.columns = df.columns.str.lower().str.strip()
    return df

final_df = clean_columns(df2)

print("Shape of dataset:", final_df.shape)      # (rows, columns)
print("\nData types:\n", final_df.dtypes)

## 2. Convert dates

In [ ]:
# Missing values before conversion
print(final_df[["start_date", "end_date"]].isnull().sum())

# Text -> datetime
final_df["start_date"] = pd.to_datetime(final_df["start_date"], errors="coerce")
final_df["end_date"] = pd.to_datetime(final_df["end_date"], errors="coerce")

final_df.info()

In [ ]:
# Missing values in all columns
print("Missing values count:\n", final_df.isnull().sum())

In [ ]:
# Check brands
print(final_df["brand"].value_counts())
print(f"\nNumber of unique brands: {final_df['brand'].nunique()}")

## 3. Create revenue column

In [ ]:
# Revenue = units sold x price
final_df["revenue_usd"] = final_df["units_sold"] * final_df["price_usd"]
print(final_df[["units_sold", "price_usd", "revenue_usd"]].head())

## 4. Create country column

In [ ]:
# Region has no missing values
print(final_df["region"].unique())
print(f"Missing values in region: {final_df['region'].isnull().sum()}")

print(final_df["city"].unique())

In [ ]:
# City -> country
city_to_country = {
    "Miami": "USA", "New York": "USA", "Los Angeles": "USA", "Chicago": "USA",
    "Berlin": "Germany", "London": "UK", "Madrid": "Spain", "Tokyo": "Japan",
    "Toronto": "Canada", "Buenos Aires": "Argentina", "Seoul": "South Korea",
    "Riyadh": "Saudi Arabia", "Jeddah": "Saudi Arabia", "Shanghai": "China",
    "Hong Kong": "China", "Lima": "Peru", "Milan": "Italy", "São Paulo": "Brazil",
    "Bogotá": "Colombia", "Singapore": "Singapore", "Doha": "Qatar",
    "Dubai": "UAE", "Abu Dhabi": "UAE", "Paris": "France", "Mexico City": "Mexico",
}
final_df["country"] = final_df["city"].map(city_to_country)

print(final_df[["city", "country"]].head(10))
# Rows with a missing city stay missing (47 rows)
print(f"Missing values in country: {final_df['country'].isnull().sum()}")

## 5. Fill missing end dates

In [ ]:
# end_date = start_date + lease_length_days
calculated_end_date = final_df["start_date"] + pd.to_timedelta(final_df["lease_length_days"], unit="days")
final_df["end_date"] = final_df["end_date"].fillna(calculated_end_date)

print("Missing values count:\n", final_df.isnull().sum())

## 6. Create category column

In [ ]:
# Product categories that appear in product names
# (checked in the order below; the first match is used)
category_list = ["Blush", "Highlighter", "Foundation", "Powder", "Eyeshadow",
                 "Lipstick", "Primer", "Mascara", "Serum", "Fragrance"]

In [ ]:
def assign_category_by_containment(product_name, category_list):
    """Return the first category whose name appears in the product name."""
    # Compare in lowercase
    product_name_lower = str(product_name).lower()
    for category in category_list:
        if category.lower() in product_name_lower:
            return category
    # No match
    return "Other"

final_df["category"] = final_df["product_name"].apply(lambda x: assign_category_by_containment(x, category_list))

print(final_df["category"].value_counts())

## 7. Save

In [ ]:
final_df.to_csv(OUTPUT, index=False, encoding="utf-8")
print(f"Saved: {OUTPUT} {final_df.shape}")